In [3]:
# 1. Install system dependencies for PDF processing
!apt-get install -y poppler-utils
!pip install bitsandbytes accelerate chandra-ocr[hf]
# 2. Install Chandra OCR
!pip install chandra-ocr[hf]


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (1,735 kB/s)     
Selecting previously unselected package poppler-utils.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━

In [4]:
from chandra.model import InferenceManager
from chandra.model.schema import BatchInputItem
from PIL import Image
import torch
import gc
from transformers import BitsAndBytesConfig, AutoModelForImageTextToText, AutoProcessor
from google.colab import userdata

# 1. Force clear GPU memory
if 'manager' in locals():
    del manager
gc.collect()
torch.cuda.empty_cache()

# 2. Get HF token from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
    print('Warning: HF_TOKEN not found in Secrets. Ensure you have added it and enabled notebook access.')

# 3. Define 4-bit config to save memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

# 4. Initialize Manager with the correct model ID
model_id = 'datalab-to/chandra-ocr-2'

print(f'Loading {model_id} in 4-bit...')
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map='auto',
    trust_remote_code=True,
    token=hf_token
)
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True, token=hf_token)

# Manually assemble the manager
manager = InferenceManager(method='hf')
manager.model = model
manager.processor = processor
print('Model loaded successfully!')

Loading datalab-to/chandra-ocr-2 in 4-bit...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.6G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/724 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/724 [00:00<?, ?it/s]

Model loaded successfully!


In [6]:
# 3. Process your worksheet
# Ensure the model has the processor attached as expected by the library
if not hasattr(manager.model, "processor"):
    manager.model.processor = manager.processor

img = Image.open("/kaggle/input/datasets/aryannsaraf/knknlknijbn/WhatsApp Image 2026-04-20 at 15.02.35.jpeg")
batch = [BatchInputItem(image=img, prompt_type="ocr_layout")]

results = manager.generate(batch)
print(results[0].markdown)
print(results)

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


# Markov chain.

(Memory less.).

(Markov property):

the probability of future actions are not dependent upon the steps that led up to the present state.

```

graph LR
    A((A)) -- 0.3 --> A
    A -- 0.7 --> B((B))
    B -- 0.8 --> A
    B -- 0.2 --> B
  
```

time-hong

probab that process beginning of A will be on B after 2 moves.

$$\begin{aligned}
 A \rightarrow A & 0.3 \times 0.7 + 0.7 \times 0.2 \rightarrow B \\
 A \rightarrow B & = 0.25 \qquad B \rightarrow B
 \end{aligned}$$
[BatchOutputItem(markdown='# Markov chain.\n\n(Memory less.).\n\n(Markov property):\n\nthe probability of future actions are not dependent upon the steps that led up to the present state.\n\n```\n\ngraph LR\n    A((A)) -- 0.3 --> A\n    A -- 0.7 --> B((B))\n    B -- 0.8 --> A\n    B -- 0.2 --> B\n  \n```\n\ntime-hong\n\nprobab that process beginning of A will be on B after 2 moves.\n\n$$\\begin{aligned}\n A \\rightarrow A & 0.3 \\times 0.7 + 0.7 \\times 0.2 \\rightarrow B \\\\\n A \\rightarrow B & = 0.25